# HoTHP vs RoTHP: Temporal Horizon Extrapolation

**Run on Colab: Runtime → Change runtime type → T4 GPU**

---

## The hypothesis

HoTHP's hyperbolic attention kernel decays **monotonically** with temporal lag by construction:

```
score(i,j) = exp(-|t_i - t_j| * θ') * f(t_i - t_j, θ)
```

where `θ' > max(θ_j)` guarantees the score decreases for any lag, no matter how large.  
The model has never seen lag=300 during training, but the architecture enforces the right answer: low weight.

RoTHP's rotary kernel is **sinusoidal**. At lags it has never seen, the cosine/sine frequencies oscillate  
unpredictably. There is no structural guarantee that distant events receive low attention.

---

## Experiment design (pre-registered)

| Split | Max events per sequence | Max normalised lag | Status |
|---|---|---|---|
| Train / Val / Test-short | `TRAIN_LEN` (=50) | ≤ 49 | in-distribution |
| Test-extrap-2x | max 100 events | ≤ 99 | lag 50–99 OOD |
| Test-extrap-5x | max 250 events | ≤ 249 | lag 50–249 OOD |
| Test-extrap-10x | max 500 events | ≤ 499 | lag 50–499 OOD |

**Two pre-specified metrics (defined before running any experiment):**

1. `overall_nll` — NLL over all events in the long sequence.
2. `ood_nll` — NLL only for events at positions ≥ `TRAIN_LEN`.  
   These events attend back to lags > 49 — genuinely OOD for both models.

**Expected outcome (falsifiable):**  
If monotonic decay matters, `ood_nll` should degrade less for HoTHP than RoTHP,  
and the gap should grow with the extrapolation factor.

**Null result (also informative):**  
If both models degrade equally, the monotonic decay inductive bias confers no practical  
benefit for horizon extrapolation — and the search for a contribution must continue elsewhere.

In [ ]:
import os
if not os.path.exists('ufc-easytpp'):
    !git clone https://github.com/hugoramos/ufc-easytpp.git
!pip install omegaconf -q

In [ ]:
import os, sys, math, random, hashlib, contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# ── Experiment parameters ─────────────────────────────────────────────────
TRAIN_LEN      = 50    # max events in training sequences
EXTRAP_FACTORS = [2, 5, 10]  # test-extrap max_events = TRAIN_LEN * factor
N_SEEDS        = 5     # random seeds (increase for publication)
N_RESTARTS     = 2     # restarts per seed (keeps best val)
EPOCHS         = 500
PATIENCE       = 30
BASE_SEED      = 42
# ─────────────────────────────────────────────────────────────────────────

def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def run_seed(*parts):
    key = '::'.join(map(str, parts))
    return (BASE_SEED + int(hashlib.sha256(key.encode()).hexdigest()[:8], 16)) % (2**31)

set_seed(BASE_SEED)
sns.set_theme(style='whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

sys.path.insert(0, os.path.abspath('ufc-easytpp'))

# Fix baselayer attention signature
import easy_tpp.model.torch_model.torch_baselayer as baselayer

def _attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        if mask.dim() == 3: mask = mask.unsqueeze(1)
        scores = scores.masked_fill(mask > 0, -1e4)
    p = torch.softmax(scores, dim=-1)
    if dropout is not None: p = dropout(p)
    return torch.matmul(p, value), p

baselayer.attention = _attention
import easy_tpp.model.torch_model.torch_rothp as rothp_module
rothp_module.attention = _attention

from easy_tpp.config_factory.model_config import ModelConfig
from easy_tpp.model.torch_model.torch_rothp import RoTHP
from easy_tpp.model.torch_model.torch_hothp import HoTHP

# AMP setup
USE_AMP = device.type == 'cuda'
if USE_AMP:
    try:
        _autocast = lambda: torch.amp.autocast(device_type='cuda')
        _Scaler   = torch.amp.GradScaler
    except AttributeError:
        _autocast = torch.cuda.amp.autocast
        _Scaler   = torch.cuda.amp.GradScaler
else:
    _autocast = contextlib.nullcontext
    _Scaler   = None

print(f'Device: {device}  |  AMP: {USE_AMP}')
print(f'Train max events (TRAIN_LEN): {TRAIN_LEN}  →  max normalised lag ≈ {TRAIN_LEN - 1}')
for f in EXTRAP_FACTORS:
    print(f'  Extrap {f}x: max events = {TRAIN_LEN * f}  →  max lag ≈ {TRAIN_LEN * f - 1}')

## Process definition

We use a **slow-decay Hawkes process** (β_norm ≈ 0.025) as the primary test because that is where  
inductive bias matters most: influence decays slowly, so events at large lags still contribute.  
RoTHP must approximate this with sinusoids; HoTHP has it baked in.

A **fast-decay process** (β_norm ≈ 0.37) serves as a control — we expect no gap there  
since distant events are irrelevant regardless of model.

In [ ]:
NUM_TYPES = 2
PAD_ID    = NUM_TYPES

PROC_SLOW = dict(mu=np.array([0.3, 0.3]),
                 alpha=np.array([[0.008, 0.006],[0.006, 0.008]]),
                 beta=0.02, label='slow-decay (β_norm≈0.025)')

PROC_FAST = dict(mu=np.array([0.4, 0.4]),
                 alpha=np.array([[0.12, 0.08],[0.08, 0.12]]),
                 beta=0.5, label='fast-decay (β_norm≈0.37)')


def simulate_hawkes(rng, proc, horizon, min_ev, max_ev):
    mu, alpha, beta = proc['mu'], proc['alpha'], proc['beta']
    for _ in range(100):
        events, t = [], 0.0
        while t < horizon and len(events) < max_ev:
            lam = mu.copy()
            for ti, ki in events:
                lam += alpha[:, ki] * np.exp(-beta * (t - ti))
            lam_bar = float(lam.sum())
            if lam_bar < 1e-9: break
            t += rng.exponential(1.0 / lam_bar)
            if t >= horizon: break
            cand = mu.copy()
            for ti, ki in events:
                cand += alpha[:, ki] * np.exp(-beta * (t - ti))
            if rng.uniform() <= cand.sum() / lam_bar:
                probs = cand / cand.sum()
                events.append((t, int(rng.choice(len(mu), p=probs))))
        if len(events) >= min_ev:
            return events[:max_ev]
    return events[:max_ev]


def make_split(rng, proc, n, horizon, max_ev, min_ev=10):
    return [simulate_hawkes(rng, proc, horizon, min_ev, max_ev) for _ in range(n)]


# Estimate beta_norm for each process
rng0 = np.random.default_rng(BASE_SEED)
for proc in [PROC_SLOW, PROC_FAST]:
    sample = make_split(rng0, proc, 200, horizon=50.0, max_ev=200, min_ev=10)
    gaps = []
    for seq in sample:
        ts = sorted([t for t, _ in seq])
        gaps.extend([ts[i]-ts[i-1] for i in range(1, len(ts))])
    mg = np.mean(gaps)
    print(f"{proc['label']}: mean_gap={mg:.3f}, β_norm={proc['beta']*mg:.4f}, "
          f"influence@lag50={math.exp(-proc['beta']*mg*50):.3f}")

In [ ]:
def to_tensors(seqs):
    """Per-sequence normalisation: mean inter-event gap = 1.0.
    After this, position i has normalised time ≈ i, and the max lag ≈ len(seq)-1.
    """
    out = []
    for seq in seqs:
        seq = sorted(seq, key=lambda x: x[0])
        t = torch.tensor([x[0] for x in seq], dtype=torch.float32)
        k = torch.tensor([x[1] for x in seq], dtype=torch.long)
        d = torch.zeros_like(t)
        d[1:] = t[1:] - t[:-1]
        mg = d[1:].mean().clamp(min=1e-6)
        t = (t - t[0]) / mg
        d = d / mg
        out.append({'time_seqs': t, 'time_delta_seqs': d, 'type_seqs': k})
    return out


def collate(batch):
    B = len(batch)
    L = max(len(x['time_seqs']) for x in batch)
    t_pad   = torch.zeros(B, L)
    d_pad   = torch.zeros(B, L)
    k_pad   = torch.full((B, L), PAD_ID, dtype=torch.long)
    npm     = torch.zeros(B, L)
    causal  = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
    attn    = torch.ones(B, L, L, dtype=torch.bool)
    for i, item in enumerate(batch):
        sl = len(item['time_seqs'])
        t_pad[i, :sl] = item['time_seqs']
        d_pad[i, :sl] = item['time_delta_seqs']
        k_pad[i, :sl] = item['type_seqs']
        npm[i, :sl]   = 1.0
        m = causal.clone(); m[:, sl:] = True; m[sl:, :] = True
        attn[i] = m
    return t_pad, d_pad, k_pad, npm, attn


def make_loader(data, bs, shuffle=False, seed=None):
    g = None
    if shuffle and seed is not None:
        g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(data, batch_size=bs, shuffle=shuffle,
                      collate_fn=collate, generator=g)


# Generate all splits
rng = np.random.default_rng(run_seed('data'))
data = {}
for pname, proc in [('slow', PROC_SLOW), ('fast', PROC_FAST)]:
    data[pname] = {
        'train': to_tensors(make_split(rng, proc, 500, horizon=50.0,  max_ev=TRAIN_LEN)),
        'val':   to_tensors(make_split(rng, proc, 150, horizon=50.0,  max_ev=TRAIN_LEN)),
        'short': to_tensors(make_split(rng, proc, 200, horizon=50.0,  max_ev=TRAIN_LEN)),
    }
    for f in EXTRAP_FACTORS:
        seqs = make_split(rng, proc, 200,
                          horizon=50.0 * f,
                          max_ev=TRAIN_LEN * f,
                          min_ev=TRAIN_LEN + 5)
        data[pname][f'extrap_{f}x'] = to_tensors(seqs)
        lengths = [len(s) for s in seqs]
        print(f'{pname} extrap_{f}x: mean_len={np.mean(lengths):.0f}, '
              f'max_lag≈{max(lengths)-1}')

print('\nOOD fraction (events at positions >= TRAIN_LEN):')
for f in EXTRAP_FACTORS:
    split = data['slow'][f'extrap_{f}x']
    total = sum(len(s['time_seqs']) for s in split)
    ood   = sum(max(0, len(s['time_seqs']) - TRAIN_LEN) for s in split)
    print(f'  {f}x: {ood}/{total} = {100*ood/total:.0f}%')

## Model configuration and training utilities

In [ ]:
config = ModelConfig(**{
    'hidden_size': 32, 'num_layers': 2, 'num_heads': 2, 'dropout_rate': 0.1,
    'num_event_types': NUM_TYPES, 'num_event_types_pad': NUM_TYPES + 1,
    'event_pad_index': PAD_ID, 'time_emb_size': 32, 'use_ln': True,
    'gpu': 0 if torch.cuda.is_available() else -1,
    'model_id': 'HorizonExtrap',
    'thinning': {'num_sample':1,'num_exp':500,'over_sample_rate':5.0,
                 'patience_counter':5,'num_samples_boundary':5,'dtime_max':5.0,'num_step_gen':1},
    'loss_integral_num_sample_per_step': 20, 'use_mc_samples': False,
})


def eval_nll(model, dl):
    model.eval(); total_l = total_n = 0
    with torch.no_grad():
        for batch in dl:
            batch = [t.to(device) for t in batch]
            with _autocast():
                l, n = model.loglike_loss(batch)
            total_l += l.item(); total_n += n
    return total_l / (total_n + 1e-9)


def eval_ood_nll(model, dl, cutoff=TRAIN_LEN):
    """NLL only for events at positions >= cutoff.
    The model still sees the full sequence for attention context.
    """
    model.eval(); total_l = total_n = 0
    with torch.no_grad():
        for batch in dl:
            t, d, k, npm, attn = [x.to(device) for x in batch]
            ood_mask = npm.clone()
            ood_mask[:, :cutoff] = 0.0
            if ood_mask.sum() == 0: continue
            with _autocast():
                l, n = model.loglike_loss([t, d, k, ood_mask, attn])
            total_l += l.item(); total_n += n
    return total_l / (total_n + 1e-9)


def train_once(model, train_dl, val_dl, lr):
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', factor=0.5, patience=10, min_lr=1e-5)
    scaler = _Scaler(enabled=True) if USE_AMP else None
    best_val, best_state, no_imp = float('inf'), None, 0
    for ep in range(EPOCHS):
        model.train()
        for batch in train_dl:
            batch = [t.to(device) for t in batch]
            opt.zero_grad()
            with _autocast():
                l, n = model.loglike_loss(batch)
                nll = l / (n + 1e-9)
            if not torch.isnan(nll):
                if scaler:
                    scaler.scale(nll).backward()
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt); scaler.update()
                else:
                    nll.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
        val = eval_nll(model, val_dl)
        sched.step(val)
        if val < best_val - 1e-4:
            best_val = val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
        if no_imp >= PATIENCE: break
    if best_state: model.load_state_dict(best_state)
    return best_val


def train_best_of(cls, train_dl, val_dl, lr, base_seed):
    """N_RESTARTS independent runs; return model with best val NLL."""
    best_val, best_state = float('inf'), None
    for r in range(N_RESTARTS):
        set_seed(base_seed + r * 7919)
        m = cls(config).to(device)
        v = train_once(m, train_dl, val_dl, lr)
        if v < best_val:
            best_val = v
            best_state = {k: v2.cpu().clone() for k, v2 in m.state_dict().items()}
    m = cls(config).to(device)
    m.load_state_dict(best_state)
    return m, best_val


print('Config and utilities ready.')

## Main experiment

For each seed and each process (slow / fast):
1. Train both models on short sequences (max lag ≈ 49)
2. Evaluate on in-dist short test set
3. Evaluate on 2× / 5× / 10× extrapolation sets using `overall_nll` and `ood_nll`

Expected runtime: ~25–40 min on T4 GPU.

In [ ]:
seeds   = [BASE_SEED + i * 100 for i in range(N_SEEDS)]
results = []   # one dict per (seed, process, factor)

for pname in ['slow', 'fast']:
    print(f'\n{"="*60}')
    print(f'Process: {pname}')
    print('='*60)

    val_dl   = make_loader(data[pname]['val'],   64)
    short_dl = make_loader(data[pname]['short'], 64)
    extrap_dls = {f: make_loader(data[pname][f'extrap_{f}x'], 16)
                  for f in EXTRAP_FACTORS}

    for seed_idx, seed in enumerate(seeds):
        print(f'\n  Seed {seed_idx+1}/{N_SEEDS} (seed={seed})')

        train_dl = make_loader(data[pname]['train'], 64, shuffle=True,
                               seed=run_seed(pname, seed))

        rothp, rv = train_best_of(RoTHP, train_dl, val_dl, lr=1e-3,
                                   base_seed=run_seed(pname, 'rothp', seed))
        hothp, hv = train_best_of(HoTHP, train_dl, val_dl, lr=5e-4,
                                   base_seed=run_seed(pname, 'hothp', seed))

        r_short = eval_nll(rothp, short_dl)
        h_short = eval_nll(hothp, short_dl)
        print(f'    val    RoTHP={rv:.4f}  HoTHP={hv:.4f}')
        print(f'    short  RoTHP={r_short:.4f}  HoTHP={h_short:.4f}')

        for f in EXTRAP_FACTORS:
            edl   = extrap_dls[f]
            r_all = eval_nll(rothp, edl)
            h_all = eval_nll(hothp, edl)
            r_ood = eval_ood_nll(rothp, edl)
            h_ood = eval_ood_nll(hothp, edl)
            print(f'    {f}x  overall: RoTHP={r_all:.4f}  HoTHP={h_all:.4f}  '
                  f'ood: RoTHP={r_ood:.4f}  HoTHP={h_ood:.4f}')
            results.append({
                'proc': pname, 'seed': seed, 'factor': f,
                'r_short': r_short,    'h_short': h_short,
                'r_all':   r_all,      'h_all':   h_all,
                'r_ood':   r_ood,      'h_ood':   h_ood,
                'r_all_deg': r_all - r_short,
                'h_all_deg': h_all - h_short,
                'r_ood_deg': r_ood - r_short,
                'h_ood_deg': h_ood - h_short,
                'adv_all': r_all - h_all,   # positive = HoTHP wins
                'adv_ood': r_ood - h_ood,
            })

print('\nExperiment complete.')

## Results and plots

In [ ]:
from scipy import stats

df = pd.DataFrame(results)

print('=' * 65)
print('RESULTS: Horizon Extrapolation')
print('=' * 65)
print(f'OOD zone = events at positions >= {TRAIN_LEN} (lags > {TRAIN_LEN-1})')

for pname in ['slow', 'fast']:
    sub = df[df['proc'] == pname]
    print(f'\n--- {pname.upper()} process ---')
    print(f'  {"Factor":>8}  {"adv_all":>10}  {"adv_ood":>10}  {"p_ood":>8}  verdict')
    for f in EXTRAP_FACTORS:
        row     = sub[sub['factor'] == f]
        adv_all = row['adv_all'].values
        adv_ood = row['adv_ood'].values
        t_stat, p2 = stats.ttest_1samp(adv_ood, 0)
        p1 = p2/2 if t_stat > 0 else 1 - p2/2
        sig = '✓ sig' if p1 < 0.05 else ('~ marginal' if p1 < 0.10 else '✗ ns')
        winner = 'HoTHP' if adv_ood.mean() > 0 else 'RoTHP'
        print(f'  {f}x:  {adv_all.mean():>+.4f}      {adv_ood.mean():>+.4f}    p={p1:.3f}  '
              f'{sig} ({winner} better)')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for row_i, pname in enumerate(['slow', 'fast']):
    sub        = df[df['proc'] == pname]
    proc_label = PROC_SLOW['label'] if pname == 'slow' else PROC_FAST['label']
    factors    = EXTRAP_FACTORS

    # ── Panel 1: OOD NLL degradation ─────────────────────────────────────
    ax = axes[row_i, 0]
    for col, label, color in [('r_ood_deg','RoTHP','#4C72B0'),
                               ('h_ood_deg','HoTHP','#C44E52')]:
        grp = sub.groupby('factor')[col]
        m, s = grp.mean(), grp.std()
        ax.plot(factors, m.values, 'o-', color=color, lw=2, ms=7, label=label)
        ax.fill_between(factors, m-s, m+s, color=color, alpha=0.13)
    ax.axhline(0, color='gray', ls='--', alpha=0.5, label='in-dist baseline')
    ax.set_xlabel('Extrapolation factor')
    ax.set_ylabel('ΔOOD NLL (OOD − short in-dist)')
    ax.set_title(f'OOD NLL degradation\n{proc_label}')
    ax.set_xticks(factors); ax.set_xticklabels([f'{f}×' for f in factors])
    ax.legend()

    # ── Panel 2: HoTHP advantage (OOD) mean ± std ────────────────────────
    ax = axes[row_i, 1]
    grp = sub.groupby('factor')['adv_ood']
    m, s = grp.mean(), grp.std()
    colors_bar = ['#55A868' if v > 0 else '#C44E52' for v in m.values]
    ax.bar(range(len(factors)), m.values, yerr=s.values,
           color=colors_bar, capsize=5, alpha=0.8)
    ax.axhline(0, color='gray', ls='--', alpha=0.6)
    ax.set_xticks(range(len(factors)))
    ax.set_xticklabels([f'{f}×' for f in factors])
    ax.set_xlabel('Extrapolation factor')
    ax.set_ylabel('HoTHP advantage (RoTHP − HoTHP OOD NLL)')
    ax.set_title(f'HoTHP advantage in OOD zone\n{proc_label}\n(green = HoTHP wins)')

    # ── Panel 3: Per-seed OOD advantage scatter ───────────────────────────
    ax = axes[row_i, 2]
    rng_jitter = np.random.RandomState(0)
    for f_i, f in enumerate(factors):
        adv = sub[sub['factor'] == f]['adv_ood'].values
        jitter = rng_jitter.uniform(-0.15, 0.15, len(adv))
        ax.scatter([f_i]*len(adv) + jitter, adv,
                   color='#4C72B0', alpha=0.7, s=50, zorder=3)
        ax.plot([f_i-0.25, f_i+0.25], [adv.mean(), adv.mean()],
                color='black', lw=2.5, zorder=4)
    ax.axhline(0, color='gray', ls='--', alpha=0.5)
    ax.set_xticks(range(len(factors)))
    ax.set_xticklabels([f'{f}×' for f in factors])
    ax.set_xlabel('Extrapolation factor')
    ax.set_ylabel('HoTHP advantage per seed')
    ax.set_title(f'Per-seed scatter\n{proc_label}\n(bar = mean)')

plt.suptitle(
    f'Horizon Extrapolation: Train max {TRAIN_LEN} events → Test up to {TRAIN_LEN*max(EXTRAP_FACTORS)} events\n'
    f'OOD zone = events at positions ≥ {TRAIN_LEN} (lags never seen during training)',
    fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('Extrapolation_Horizon.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Attention profiles: does HoTHP stay low past the training boundary? ──────
# Re-trains one pair of models for profiling (uses last seed).

def extract_attn(model, dl, model_type, max_batches=20):
    model.eval()
    lags, weights = [], []
    with torch.no_grad():
        for b_idx, batch in enumerate(dl):
            if b_idx >= max_batches: break
            t, d, k, npm, attn = [x.to(device) for x in batch]
            enc   = model.layer_type_emb(k)
            layer = model.stack_layers[0]
            if model_type == 'rothp':
                cos, sin = model.rotary_emb(t)
                _, aw = layer.self_attn(enc, enc, enc, attn,
                                        cos=cos, sin=sin, output_weight=True)
            else:
                nt = model._normalize_timestamps(t)
                _, aw = layer.self_attn(enc, enc, enc, attn,
                                        time_seqs=nt,
                                        thetas=model.hope_emb.thetas,
                                        theta_prime=model.hope_emb.theta_prime,
                                        output_weight=True)
            aw    = aw.mean(dim=1).cpu()
            t_cpu = t.cpu(); npm_cpu = npm.cpu()
            for bi in range(t_cpu.shape[0]):
                sl = int(npm_cpu[bi].sum().item())
                for i in range(sl):
                    for j in range(i):
                        lag = float(t_cpu[bi, i] - t_cpu[bi, j])
                        if lag > 0:
                            lags.append(lag)
                            weights.append(float(aw[bi, i, j]))
    return np.array(lags), np.array(weights)


def bin_attn(lags, weights, n_bins=60):
    q98   = np.quantile(lags, 0.98)
    edges = np.linspace(0, q98, n_bins + 1)
    c     = (edges[:-1] + edges[1:]) / 2
    m, s  = [], []
    for i in range(n_bins):
        mask = (lags >= edges[i]) & (lags < edges[i+1])
        m.append(weights[mask].mean() if mask.sum() > 5 else np.nan)
        s.append(weights[mask].std()  if mask.sum() > 5 else np.nan)
    return c, np.array(m), np.array(s)


fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, pname in zip(axes, ['slow', 'fast']):
    proc_label = PROC_SLOW['label'] if pname == 'slow' else PROC_FAST['label']
    edl = make_loader(data[pname][f'extrap_{max(EXTRAP_FACTORS)}x'], 16)
    tdl = make_loader(data[pname]['train'], 64, shuffle=True,
                      seed=run_seed('profile', pname))
    vdl = make_loader(data[pname]['val'], 64)

    rothp_p, _ = train_best_of(RoTHP, tdl, vdl, lr=1e-3,
                                base_seed=run_seed('profile', pname, 'rothp'))
    hothp_p, _ = train_best_of(HoTHP, tdl, vdl, lr=5e-4,
                                base_seed=run_seed('profile', pname, 'hothp'))

    r_lags, r_w = extract_attn(rothp_p, edl, 'rothp')
    h_lags, h_w = extract_attn(hothp_p, edl, 'hothp')
    rc, rm, rs  = bin_attn(r_lags, r_w)
    hc, hm, hs  = bin_attn(h_lags, h_w)

    ax.plot(rc, rm, color='#4C72B0', lw=1.8, label='RoTHP')
    ax.fill_between(rc, rm-rs, rm+rs, color='#4C72B0', alpha=0.13)
    ax.plot(hc, hm, color='#C44E52', lw=1.8, label='HoTHP')
    ax.fill_between(hc, hm-hs, hm+hs, color='#C44E52', alpha=0.13)
    ax.axvline(TRAIN_LEN - 1, color='green', ls='--', alpha=0.7,
               label=f'Training max lag ({TRAIN_LEN-1})')
    ax.set_xlabel('Normalised temporal lag')
    ax.set_ylabel('Mean attention weight')
    ax.set_title(f'Attention profile on {max(EXTRAP_FACTORS)}× sequences\n{proc_label}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle(
    f'Does HoTHP keep attention low past the training boundary (green)?\n'
    f'RoTHP has no structural guarantee; HoTHP decays monotonically by construction.',
    fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('Attention_Profile_Extrap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from scipy import stats

print('=' * 65)
print('SUMMARY')
print('=' * 65)
print(f'Training max lag: {TRAIN_LEN - 1}  |  Seeds: {N_SEEDS}')
print()
print('adv_ood > 0  →  HoTHP lower OOD NLL  →  monotonic decay helps')
print('adv_ood ≈ 0  →  no benefit to monotonic decay for extrapolation')
print()

for pname in ['slow', 'fast']:
    sub        = df[df['proc'] == pname]
    proc_label = PROC_SLOW['label'] if pname == 'slow' else PROC_FAST['label']
    print(f'{proc_label}')
    advs = []
    for f in EXTRAP_FACTORS:
        row    = sub[sub['factor'] == f]
        adv    = row['adv_ood'].values
        t_stat, p2 = stats.ttest_1samp(adv, 0)
        p1     = p2/2 if t_stat > 0 else 1 - p2/2
        sig    = '✓' if p1 < 0.05 else ('~' if p1 < 0.10 else '✗')
        winner = 'HoTHP' if adv.mean() > 0 else 'RoTHP'
        print(f'  {f}x: adv_ood={adv.mean():+.4f} (p={p1:.3f}) {sig}  → {winner} better')
        advs.append(adv.mean())
    trend = 'INCREASING (monotonic decay helps more at larger horizons)' \
            if advs[-1] > advs[0] else 'FLAT/DECREASING (no growing benefit)'
    print(f'  Trend: {trend}')
    print()